# Advanced Pattern Matching

## Overview
This notebook covers alternation, anchors, word boundaries, and the critical PostgreSQL POSIX ERE differences from Python regex.

**PostgreSQL POSIX ERE vs Python `re` — key differences:**

| Feature | Python `re` | PostgreSQL POSIX ERE |
|---|---|---|
| Word boundary | `\b` | `\m` (start), `\M` (end), `\y` (either) |
| Digit shorthand | `\d` | `[[:digit:]]` or `[0-9]` |
| Whitespace | `\s` | `[[:space:]]` |
| Lookahead | `(?=...)` | **Not supported** |
| Lookbehind | `(?<=...)` | **Not supported** |
| Non-capturing group | `(?:...)` | PostgreSQL 15+ only |
| Case-insensitive | `(?i)` flag | Use `~*` operator |

**Negation workaround:**
```sql
WHERE note_text ~*  'allerg'
  AND note_text !~* '\m(no|not)\s+allerg'
```

---

In [1]:
import sqlite3, re

def make_conn():
    conn = sqlite3.connect(":memory:")
    conn.row_factory = sqlite3.Row
    conn.create_function("regexp", 2,
        lambda p, s: bool(re.search(p, s or "", re.IGNORECASE)))
    return conn

conn = make_conn()
conn.executescript("""
CREATE TABLE clinical_notes (
    note_id INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id TEXT NOT NULL, note_date TEXT NOT NULL, note_text TEXT NOT NULL
);
CREATE TABLE medications (
    med_id INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id TEXT NOT NULL, drug_name TEXT NOT NULL, dosage TEXT, frequency TEXT
);
INSERT INTO clinical_notes VALUES
  (NULL,'P001','2023-04-10','BP 142/88 mmHg. Prescribed Lisinopril 10mg daily. No allergies.'),
  (NULL,'P001','2023-10-01','BP 128/82 mmHg. HbA1c 7.2%. Metformin 500mg BID. No adverse effects.'),
  (NULL,'P002','2023-05-15','SpO2 94%. Salbutamol 2.5mg nebulised. Allergic to penicillin.'),
  (NULL,'P003','2023-08-20','BMI 34.2. BP 148/92 mmHg. No current medications.'),
  (NULL,'P004','2023-09-01','Healthy adult. BP 118/76 mmHg. ECG normal sinus rhythm. No complaints.'),
  (NULL,'P005','2023-11-10','HbA1c 8.9%. Insulin 20 units. Metformin 1g BD. Previous MI 2019.');
INSERT INTO medications VALUES
  (NULL,'P001','Lisinopril','10mg','once daily'),
  (NULL,'P001','Metformin','500mg','BID'),
  (NULL,'P002','Salbutamol','2.5mg','PRN'),
  (NULL,'P002','Fluticasone','250mcg','BD'),
  (NULL,'P003','Amlodipine','5mg','once daily'),
  (NULL,'P005','Insulin glargine','20units','nocte'),
  (NULL,'P001','Atorvastatin','40mg','once daily');
""")
conn.commit()
print("Advanced dataset:",
      conn.execute("SELECT COUNT(*) FROM clinical_notes").fetchone()[0], "notes,",
      conn.execute("SELECT COUNT(*) FROM medications").fetchone()[0], "meds")


Advanced dataset: 6 notes, 7 meds


---
## Alternation and frequency detection

In [2]:
import re

print("=== Alternation (|) and frequency term detection ===")
freq_pat = re.compile(r'\b(BID|BD|TID|QID|PRN|nocte|daily|once daily)\b', re.IGNORECASE)
known_pat = re.compile(r'^(BID|BD|TID|QID|once daily|PRN|nocte|daily)$', re.IGNORECASE)

rows = conn.execute("SELECT note_id, patient_id, note_text FROM clinical_notes").fetchall()
print("Notes containing frequency terms:")
for row in rows:
    matches = freq_pat.findall(row['note_text'])
    if matches:
        print(f"  note {row['note_id']} ({row['patient_id']}): {matches}")

print()
rows2 = conn.execute("SELECT med_id, drug_name, frequency FROM medications").fetchall()
print("Medication frequency classification:")
for r in rows2:
    freq   = r['frequency'] or ''
    status = 'known' if known_pat.match(freq) else 'UNRECOGNISED'
    print(f"  {r['drug_name']:<22s}  {freq!r:<14s}  {status}")

print()
print("PostgreSQL alternation:")
print("  WHERE frequency ~* '^(BID|BD|TID|QID|once daily|PRN|nocte|daily)$'")
print("  -- word boundary in PostgreSQL POSIX: \\m (start) \\M (end) not \\b")
print("  WHERE note_text ~* '\\m(BID|BD|TID|QID|PRN|nocte|daily)\\M'")


=== Alternation (|) and frequency term detection ===
Notes containing frequency terms:
  note 1 (P001): ['daily']
  note 2 (P001): ['BID']
  note 6 (P005): ['BD']

Medication frequency classification:
  Lisinopril              'once daily'    known
  Metformin               'BID'           known
  Salbutamol              'PRN'           known
  Fluticasone             'BD'            known
  Amlodipine              'once daily'    known
  Insulin glargine        'nocte'         known
  Atorvastatin            'once daily'    known

PostgreSQL alternation:
  WHERE frequency ~* '^(BID|BD|TID|QID|once daily|PRN|nocte|daily)$'
  -- word boundary in PostgreSQL POSIX: \m (start) \M (end) not \b
  WHERE note_text ~* '\m(BID|BD|TID|QID|PRN|nocte|daily)\M'


---
## Anchors and word boundaries

In [3]:
import re

print("=== Anchors and word boundaries: Python re vs PostgreSQL POSIX ===")
print()

examples = [
    (r'\bBP\b',     'BP 142/88 mmHg. BP elevated.',  True,  'standalone BP matches'),
    (r'\bBP\b',     'no BPM data available',           False, 'BP inside BPM should NOT match'),
    (r'^BP',        'BP 142/88 mmHg',                  True,  'BP at string start'),
    (r'^BP',        'Patient BP 142/88',                False, 'BP not at start'),
    (r'mmHg$',      'BP 142/88 mmHg',                  True,  'mmHg at string end'),
    (r'mmHg$',      'BP 142/88 mmHg elevated',         False, 'mmHg not at end'),
    (r'(?i)\bno\b', 'No current medications.',          True,  'case-insensitive no'),
    (r'(?i)\bno\b', 'Note: normal values',              False, 'no should not match inside Note'),
]

print(f"  {'Pattern':<18s}  {'Match?':<9s}  {'Correct?':<10s}  Note")
print("  " + "-"*72)
for pattern, string, expected, note in examples:
    actual  = bool(re.search(pattern, string))
    correct = 'YES' if actual == expected else 'WRONG'
    result  = 'match    ' if actual else 'no match '
    print(f"  {pattern!r:<18s}  {result}  {correct:<10s}  {note}")

print()
print("PostgreSQL POSIX ERE word boundary (different from Python \\b):")
table = [
    ('\\m',  'Word START boundary  (PostgreSQL POSIX only)'),
    ('\\M',  'Word END boundary    (PostgreSQL POSIX only)'),
    ('\\y',  'Word boundary either side (PostgreSQL POSIX)'),
    ('\\b',  'NOT a word boundary in PostgreSQL ERE -- use \\m/\\M'),
    ('\\d',  'NOT in PostgreSQL ERE -- use [0-9] or [[:digit:]]'),
    ('\\s',  'NOT in PostgreSQL ERE -- use [[:space:]]'),
    ('\\w',  'NOT in PostgreSQL ERE -- use [[:alnum:]_]'),
]
for sym, desc in table:
    print(f"  {sym!r:<8s}  {desc}")


=== Anchors and word boundaries: Python re vs PostgreSQL POSIX ===

  Pattern             Match?     Correct?    Note
  ------------------------------------------------------------------------
  '\\bBP\\b'          match      YES         standalone BP matches
  '\\bBP\\b'          no match   YES         BP inside BPM should NOT match
  '^BP'               match      YES         BP at string start
  '^BP'               no match   YES         BP not at start
  'mmHg$'             match      YES         mmHg at string end
  'mmHg$'             no match   YES         mmHg not at end
  '(?i)\\bno\\b'      match      YES         case-insensitive no
  '(?i)\\bno\\b'      no match   YES         no should not match inside Note

PostgreSQL POSIX ERE word boundary (different from Python \b):
  '\\m'     Word START boundary  (PostgreSQL POSIX only)
  '\\M'     Word END boundary    (PostgreSQL POSIX only)
  '\\y'     Word boundary either side (PostgreSQL POSIX)
  '\\b'     NOT a word boundary in Po

---
## Negation context: lookahead workarounds

In [4]:
import re

print("=== Negation context: 'no allergies' vs 'allergic to ...' ===")
print("PostgreSQL has NO lookahead. Workaround: combine ~ with !~")
print()

rows = conn.execute("SELECT note_id, patient_id, note_text FROM clinical_notes").fetchall()
for row in rows:
    text = row['note_text']
    has_allergy  = bool(re.search(r'allerg', text, re.IGNORECASE))
    has_negation = bool(re.search(r'\b(no|not)\s+allerg', text, re.IGNORECASE))
    if has_allergy:
        label = 'NEGATED  ' if has_negation else 'POSITIVE '
        print(f"  note {row['note_id']} {label}: {text[:65]}...")

print()
print("PostgreSQL equivalent (no lookahead -- use AND/NOT combination):")
print("""
  -- Positive allergy (not negated):
  WHERE note_text ~*  'allerg'
    AND note_text !~* '\\m(no|not)\\s+allerg'

  -- Negated allergy:
  WHERE note_text ~* '\\m(no|not)\\s+allerg'
""")
print("Notes on PostgreSQL lookahead:")
for n in [
    "PostgreSQL POSIX ERE does NOT support (?=), (?!), (?<=), (?<!)",
    "Workaround: combine ~ (match) with !~ (exclusion) using AND",
    "For full PCRE support: use PL/Python or PL/Perl UDF",
    "For clinical NLP negation: use application-layer (spaCy, medspaCy)",
    "PostgreSQL 15+ added (?:) non-capturing groups but not lookahead",
]:
    print(f"  - {n}")


=== Negation context: 'no allergies' vs 'allergic to ...' ===
PostgreSQL has NO lookahead. Workaround: combine ~ with !~

  note 1 NEGATED  : BP 142/88 mmHg. Prescribed Lisinopril 10mg daily. No allergies....
  note 3 POSITIVE : SpO2 94%. Salbutamol 2.5mg nebulised. Allergic to penicillin....

PostgreSQL equivalent (no lookahead -- use AND/NOT combination):

  -- Positive allergy (not negated):
  WHERE note_text ~*  'allerg'
    AND note_text !~* '\m(no|not)\s+allerg'

  -- Negated allergy:
  WHERE note_text ~* '\m(no|not)\s+allerg'

Notes on PostgreSQL lookahead:
  - PostgreSQL POSIX ERE does NOT support (?=), (?!), (?<=), (?<!)
  - Workaround: combine ~ (match) with !~ (exclusion) using AND
  - For full PCRE support: use PL/Python or PL/Perl UDF
  - For clinical NLP negation: use application-layer (spaCy, medspaCy)
  - PostgreSQL 15+ added (?:) non-capturing groups but not lookahead


---
## Common Pitfalls

**1. Using Python `\b` in PostgreSQL**
`\b` is not a word boundary in PostgreSQL POSIX ERE. Use `\m`/`\M`/`\y` instead. `WHERE note ~ '\bBP\b'` silently fails to enforce the boundary.

**2. Using `\d` and `\s` in PostgreSQL**
Use `[0-9]` or `[[:digit:]]` for digits and `[[:space:]]` for whitespace in PostgreSQL ERE.

**3. Attempting lookahead in PostgreSQL**
`(?=...)`, `(?!...)` are PCRE features not in PostgreSQL POSIX ERE. Workaround: `WHERE col ~* 'pattern' AND col !~* 'exclusion'`.

**4. Greedy `.*` consuming too much**
`regexp_match(col, 'BP (.*)mmHg')` on two BP readings returns everything up to the LAST `mmHg`. Use specific character classes: `'BP ([0-9/]+)\s*mmHg'`.

**5. Alternation precedence surprises**
`cat|dog food` means `cat` OR `dog food`. Always group: `(cat|dog) food`.

**6. `~` is case-sensitive in PostgreSQL**
`WHERE frequency ~ '^bid$'` does not match `'BID'`. Use `~*` for case-insensitive matching.


---
*sql_methods_library - Samantha McGarrigle*